In [7]:
from ortools.linear_solver import pywraplp
import os

# 1. ENTRADA DE DADOS
print("Dados do Problema:")
print("Como deseja inserir os dados?")
print("1 - Manualmente")
print("2 - Ler arquivo dados.txt")
opcao = input("Escolha uma opção (1 Ler ou 2 Arquivo): ")

itens = []
demandas = []

if opcao == '2':
    filename = "dados.txt"
    if os.path.exists(filename):
        try:
            with open(filename, 'r') as f:
                # Remove linhas vazias e espaços extras
                lines = [line.strip() for line in f.readlines() if line.strip()]
                
                # Extrai os valores após o caractere ':'
                L = int(lines[0].split(':')[1].strip())
                num_tipos = int(lines[1].split(':')[1].strip())
                
                for i in range(2, 2 + num_tipos):
                    # Procura pelos valores após 'comprimento:' e 'demanda:'
                    linha = lines[i]
                    if 'comprimento:' in linha and 'demanda:' in linha:
                        tamanho = int(linha.split('comprimento:')[1].split('demanda:')[0].strip())
                        qtd = int(linha.split('demanda:')[1].strip())
                        itens.append(tamanho)
                        demandas.append(qtd)
            print(f"Dados carregados de {filename} com sucesso!")
        except Exception as e:
            print(f"Erro ao ler o arquivo: {e}")
            opcao = '1'
    else:
        print("Arquivo 'dados.txt' não encontrado. Prosseguindo com entrada manual...")
        opcao = '1'

if opcao == '1':
    L = int(input("Tamanho da barra (Ex: 150): "))
    num_tipos = int(input("Quantidade de tipos de itens (Ex: 3): "))

    for i in range(num_tipos):
        tamanho = int(input(f"Tamanho do item {i+1}: "))
        qtd = int(input(f"Demanda do item {i+1}: "))
        itens.append(tamanho)
        demandas.append(qtd)

Dados do Problema:
Como deseja inserir os dados?
1 - Manualmente
2 - Ler arquivo dados.txt
Dados carregados de dados.txt com sucesso!


In [8]:
# 2. GERAÇÃO DE PADRÕES DE CORTE
padroes = []

def gerar_padroes(i, gasto_atual, padrao_atual):
    if i == num_tipos:
        # Verifica se o padrão ta cheio
        pode_adicionar_mais = False
        for item in itens:
            if gasto_atual + item <= L:
                pode_adicionar_mais = True
                break

        if not pode_adicionar_mais and sum(padrao_atual) > 0:
            padroes.append(list(padrao_atual))
        return

    # Máximo de vezes que o item atual pode aparecer
    max_qtd = (L - gasto_atual) // itens[i]
    for qtd in range(int(max_qtd) + 1):
        padrao_atual[i] = qtd
        gerar_padroes(i + 1, gasto_atual + qtd * itens[i], padrao_atual)

print("Gerando padrões de corte válidos...")
gerar_padroes(0, 0, [0] * num_tipos)
print(f"Total de padrões gerados: {len(padroes)}")

Gerando padrões de corte válidos...
Total de padrões gerados: 23


In [9]:
# 3. CRIAÇÃO DO SOLVER E MODELAGEM
solver = pywraplp.Solver.CreateSolver('SCIP')
infinity = solver.infinity()

# 4. VARIÁVEIS DE DECISÃO
# x[j] = quantidade de vezes que o padrão j será utilizado
x = {}
for j in range(len(padroes)):
    x[j] = solver.IntVar(0, infinity, f'x_{j}')

# 5. FUNÇÃO OBJETIVO é minimizar o total de barras brutas utilizadas
objetivo = solver.Objective()
for j in range(len(padroes)):
    objetivo.SetCoefficient(x[j], 1)
objetivo.SetMinimization()

# 6. RESTRIÇÕES
for i in range(num_tipos):
    # Soma(qtd_item_no_padrao * vezes_que_usa_padrao) >= demanda
    restricao = solver.Constraint(demandas[i], infinity, f'resto_demanda_item_{itens[i]}')
    for j in range(len(padroes)):
        if padroes[j][i] > 0:
            restricao.SetCoefficient(x[j], padroes[j][i])

# 7. SOLVE
status = solver.Solve()

In [10]:
# 8. RESULTADOS
if status == pywraplp.Solver.OPTIMAL:
    total_barras = int(solver.Objective().Value())
    print(f"SOLUÇÃO ÓTIMA ENCONTRADA")
    print(f"Total de barras de {L}m utilizadas: {total_barras}")

    # Cálculo do Desperdício (Total comprado - Total entregue)
    comprimento_utilizado = sum(itens[i] * demandas[i] for i in range(num_tipos))
    desperdicio_total = (total_barras * L) - comprimento_utilizado
    print(f"Desperdício total: {desperdicio_total}m")

    print("\nPadrões de corte ativos:")
    for j in range(len(padroes)):
        qtd_uso = int(x[j].solution_value())
        if qtd_uso > 0:
            detalhe_corte = ", ".join([f"{padroes[j][k]}x({itens[k]}m)" for k in range(num_tipos) if padroes[j][k] > 0])
            print(f"- Padrão {j} [{detalhe_corte}]: usado {qtd_uso} vezes")

    # 9. EXPORTAÇÃO DO MODELO LP
    print("\n--- MODELO MATEMÁTICO (LP FORMAT) ---")
    print(solver.ExportModelAsLpFormat(False))
else:
    print("O solver não conseguiu encontrar uma solução ótima.")


SOLUÇÃO ÓTIMA ENCONTRADA
Total de barras de 150m utilizadas: 11
Desperdício total: 0m

Padrões de corte ativos:
- Padrão 2 [2x(60m), 6x(5m)]: usado 2 vezes
- Padrão 7 [3x(45m), 3x(5m)]: usado 4 vezes
- Padrão 15 [2x(30m), 1x(60m), 6x(5m)]: usado 1 vezes
- Padrão 17 [2x(30m), 2x(45m)]: usado 4 vezes

--- MODELO MATEMÁTICO (LP FORMAT) ---
\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 4
\   Variables        : 23
\     Binary         : 0
\     Integer        : 23
\     Continuous     : 0
Minimize
 Obj: +1 x_0 +1 x_1 +1 x_2 +1 x_3 +1 x_4 +1 x_5 +1 x_6 +1 x_7 +1 x_8 +1 x_9 +1 x_10 +1 x_11 +1 x_12 +1 x_13 +1 x_14 +1 x_15 +1 x_16 +1 x_17 +1 x_18 +1 x_19 +1 x_20 +1 x_21 +1 x_22 
Subject to
 resto_demanda_item_30: +1 x_8 +1 x_9 +1 x_10 +1 x_11 +1 x_12 +1 x_13 +2 x_14 +2 x_15 +2 x_16 +2 x_17 +3 x_18 +3 x_19 +3 x_20 +4 x_21 +5 x_22  >= 10
 resto_demanda_item_45: +1 x_3 +1 x_4 +2 x_5 +2 x_6 +3 x_7 +1 x_11 +1 x_12 +2 x_13 +1 x_16 +2 x_